# NumCompute Quickstart: Exploring Titanic Passengers

Instead of working with a table of unexplained prices, this quickstart uses passenger records with familiar fields such as age, fare, sex, and survival. The CSV contains real missing values, so we can see why preprocessing is useful.

We will load and inspect the records, summarize and visualize passenger ages, then use a pipeline to fill in missing ages/fares and put them on a comparable scale. We will also encode a category and evaluate a simple, clearly stated rule. At the end, short examples will introduce the rest of NumCompute: streaming, ranking, sorting, numeric utilities, derivatives, and benchmarking.

Dataset: [Seaborn Titanic dataset](https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv), included locally as `demo/data/titanic.csv` so the notebook does not need to download data while running.

The setup cell finds the project folder and imports NumCompute directly from its source; it does not depend on a `.venv` folder. The selected Python kernel still needs NumPy installed.

In [1]:
from pathlib import Path
import sys
import numpy as np

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "NumCompute" / "numcompute").is_dir()),
    Path.cwd(),
)
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "NumCompute"))

DATA_PATH = project_root / "demo" / "data" / "titanic.csv"
print("Dataset:", DATA_PATH)

from numcompute.io import iter_csv, load_csv
from numcompute.rank import Ranker


Dataset: /Users/prajalpatidar/Desktop/COMP-5004/Morpheus-Assignment-2.1-Programming-Task-1-/demo/data/titanic.csv


## 1. Load the passenger records

`load_csv` returns the column names and the table. We ask it to represent blank cells as `NaN` (not a made-up value). `NaN` means the original record did not include that information.

In [ ]:
columns, passengers = load_csv(DATA_PATH, fill_value=np.nan)
column_index = {name: index for index, name in enumerate(columns)}

print(f"Loaded {passengers.shape[0]} passenger records.")
print("Available columns:", ", ".join(columns))

shown_columns = ["survived", "pclass", "sex", "age", "fare", "embarked"]
print("\nFirst five passengers (survived: 1 = yes, 0 = no):")
for row in passengers[:5]:
    print({name: row[column_index[name]] for name in shown_columns})

Loaded 891 passenger records.
Available columns: survived, pclass, sex, age, sibsp, parch, fare, embarked, class, who, adult_male, deck, embark_town, alive, alone

First five passengers (survived: 1 = yes, 0 = no):
{'survived': 0.0, 'pclass': 3.0, 'sex': 'male', 'age': 22.0, 'fare': 7.25, 'embarked': 'S'}
{'survived': 1.0, 'pclass': 1.0, 'sex': 'female', 'age': 38.0, 'fare': 71.2833, 'embarked': 'C'}
{'survived': 1.0, 'pclass': 3.0, 'sex': 'female', 'age': 26.0, 'fare': 7.925, 'embarked': 'S'}
{'survived': 1.0, 'pclass': 1.0, 'sex': 'female', 'age': 35.0, 'fare': 53.1, 'embarked': 'S'}
{'survived': 0.0, 'pclass': 3.0, 'sex': 'male', 'age': 35.0, 'fare': 8.05, 'embarked': 'S'}


## 9. Rank, sort, and search passenger values

These functions answer related but different questions: sorting returns values in order, ranking assigns a position to every value, and quickselect finds one requested position without sorting the whole list. `topk` selects a few extremes (without promising to sort that selection), while binary search finds an insertion point in an already sorted array. The multi-key example orders a small passenger sample by ticket class, then fare.

In [ ]:
## Rank values

`Ranker` assigns positions to values and can handle ties using average ranks. This small example checks ranking and percentile behaviour on a known sample.

Sorted ages: [22. 22. 28. 35. 35. 38.]
Average ranks (ties share a rank): [1.5 6.  1.5 4.5 3.  4.5]
Median from ranker: 31.5
Third-smallest age: 28.0
Two largest ages: [35. 38.]
Where would age 30 be inserted? (3, False)
First eight passenger rows ordered by class, then fare: [6 3 1 0 2 4 5 7]


example_ages = np.array([22.0, 38.0, 22.0, 35.0, 28.0, 35.0])
ranker = Ranker()

print("Average ranks (ties share a rank):", ranker.rank(example_ages, method="average"))
print("Median from ranker:", ranker.percentile(example_ages, 50))